# Stage 2 Signature Detection Notes

This notebook is for checking the Stage 2 flow-based signature detection work.

Stage 2 is not a machine learning stage. It uses hand-written flow rules to catch obvious attack patterns from the processed CSE-CIC-IDS2018 flow features. The point here is to keep the detection logic easy to read and easy to explain before combining it with the Stage 3 ML model later.

The important rule for this stage is simple: detection uses feature-only input. Ground truth is only used after prediction, when I need to evaluate the result.

## 1. Paths and Setup

These are the main Stage 2 files I need to check.

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'stage-2').exists():
    REPO_ROOT = REPO_ROOT.parent

FLOW_FEATURE_PATH = REPO_ROOT / 'stage-1' / 'data' / 'processed' / 'flow-feature-sample.csv'
GROUND_TRUTH_PATH = REPO_ROOT / 'stage-1' / 'data' / 'processed' / 'ground-truth.json'
RULES_PATH = REPO_ROOT / 'stage-2' / 'rules' / 'flow-signatures.json'
SIGNATURE_OUTPUT_PATH = REPO_ROOT / 'stage-2' / 'data' / 'signature-output.sample.json'
EVALUATION_SUMMARY_PATH = REPO_ROOT / 'stage-2' / 'evaluation' / 'signature-evaluation-summary.json'
AUDIT_DIR = REPO_ROOT / 'stage-2' / 'audit'

for path in [FLOW_FEATURE_PATH, GROUND_TRUTH_PATH, RULES_PATH]:
    print(path, 'exists=' + str(path.exists()))

## 2. Load Feature-Only Input

This is the input used by the signature engine. It should not contain answer fields like `attackType` or `groundTruth`.

In [ ]:
flow_df = pd.read_csv(FLOW_FEATURE_PATH)
print('Rows:', len(flow_df))
print('Columns:', list(flow_df.columns))

forbidden_fields = {'Label', 'rawLabel', 'attackType', 'mappedAttackType', 'groundTruth', 'severity', 'similarityKey'}
leaked_fields = sorted(forbidden_fields.intersection(flow_df.columns))
print('Forbidden fields found:', leaked_fields if leaked_fields else 'none')

## 3. Check Signature Rules

The rules are stored as JSON so they can be inspected without reading source code. I mainly check what each rule predicts and which flow features it uses.

In [ ]:
with open(RULES_PATH, 'r', encoding='utf-8') as file:
    rules = json.load(file)

print('Rule count:', len(rules))
for rule in rules:
    print(f"{rule['id']} -> {rule['predictedAttackType']}")
    print('  conditions:', rule['condition'])

## 4. Run Signature Demo

The actual Stage 2 engine is written in JavaScript. From the repository root, run:

```powershell
node stage-2/scripts/run-signature-demo.js
```

This produces signature output and evaluation files. If I am using Codex bundled Node or local Node, the command is the same once `node` is available.

In [ ]:
# This cell is optional. It only works when Node.js is available in the notebook/runtime environment.
# !node stage-2/scripts/run-signature-demo.js

## 5. Inspect Signature Output

The signature output should contain the original flow ID plus the signature result. It should show whether a rule matched and what attack type the rule predicted.

In [ ]:
if SIGNATURE_OUTPUT_PATH.exists():
    with open(SIGNATURE_OUTPUT_PATH, 'r', encoding='utf-8') as file:
        signature_output = json.load(file)
    print('Signed records:', len(signature_output))
    print(signature_output[0])
else:
    print('Signature output not found yet. Run the Stage 2 demo first.')

## 6. Review Evaluation Summary

Ground truth is joined only after prediction. This section is for checking the measured result, not for creating the prediction.

In [ ]:
if EVALUATION_SUMMARY_PATH.exists():
    with open(EVALUATION_SUMMARY_PATH, 'r', encoding='utf-8') as file:
        evaluation = json.load(file)
    print(json.dumps(evaluation, indent=2)[:4000])
else:
    print('Evaluation summary not found yet. Run the Stage 2 demo first.')

## 7. Stage 2B Audit Files

Stage 2B is my rule checking step. It does not tune thresholds. It just helps me see whether the current rules make sense on the current sample.

In [ ]:
for path in sorted(AUDIT_DIR.glob('*')):
    print(path.name, path.stat().st_size, 'bytes')

## 8. Notes for Report

Short wording I can reuse later:

- Stage 2 uses flow-based signature rules instead of packet payload inspection.
- The rules use observable fields such as protocol, destination port, packet rate, packet counts, duration, and byte rate.
- `attackType` and `groundTruth` are not used during signature matching.
- Ground truth is joined after prediction for evaluation only.
- The current rules are prototype heuristics. They are useful for explanation and for the later fusion engine, but they are not claimed to be production IDS rules.

## 9. Current Limitations

- Flow records do not include payload, so the engine cannot inspect URLs, SQL strings, login failures, or malware signatures directly.
- Some attack types are hard to prove from one flow row alone. DDoS and infiltration especially need more context in a real system.
- Stage 2 should be combined with Stage 3 ML output later instead of being treated as the whole detection system.